# 07 - Predictive Modeling

## Customer Intelligence Platform

This notebook trains 5 classification models, compares their performance,
and performs threshold optimization to maximize business value.

### Models
1. Logistic Regression (interpretable baseline)
2. Random Forest (tree-based baseline)
3. XGBoost (gradient boosting)
4. LightGBM (fast gradient boosting)
5. CatBoost (categorical-native boosting)

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib

from src.load_data import load_features
from src.preprocessing import prepare_data, encode_features, split_data, scale_features
from src.train import get_models, train_model, train_all_models, save_models
from src.evaluate import (
    compute_metrics,
    plot_confusion_matrix,
    plot_roc_curves,
    plot_precision_recall_curves,
    plot_threshold_analysis,
    optimize_threshold,
    model_comparison_table,
)
from src.config import TARGET, BEST_MODEL_FILE, MODELS_DIR

%matplotlib inline


In [ ]:
df = load_features()


## 1. Preprocessing


In [ ]:
X_train, X_test, y_train, y_test, scaler, feature_names = prepare_data(df)
print(f"\nFeature names ({len(feature_names)}):")
for i, f in enumerate(feature_names[:10]):
    print(f"  {i+1}. {f}")
print(f"  ... and {len(feature_names)-10} more")


## 2. Train All Models


In [ ]:
results, best_name, best_model = train_all_models(X_train, X_test, y_train, y_test)


## 3. Model Comparison


In [ ]:
comparison = model_comparison_table(results)
comparison


## 4. ROC Curves


In [ ]:
models_dict = {r["name"]: r["model"] for r in results}
fig = plot_roc_curves(models_dict, X_test, y_test)
plt.show()


## 5. Precision-Recall Curves


In [ ]:
fig = plot_precision_recall_curves(models_dict, X_test, y_test)
plt.show()


## 6. Best Model - Detailed Evaluation


In [ ]:
y_prob = best_model.predict_proba(X_test)[:, 1]
y_pred = best_model.predict(X_test)

print(f"Best Model: {best_name}")
print(f"\n{'='*40}")
metrics = compute_metrics(y_test, y_pred, y_prob)
for k, v in metrics.items():
    print(f"  {k}: {v}")


In [ ]:
fig = plot_confusion_matrix(y_test, y_pred, title=f"Confusion Matrix - {best_name}")
plt.show()


## 7. Threshold Optimization

Most student projects use the default 0.5 threshold.
We optimize the threshold to maximize F1 score.


In [ ]:
fig = plot_threshold_analysis(y_test, y_prob)
plt.show()


In [ ]:
opt_result = optimize_threshold(y_test, y_prob, metric="f1")
print(f"Optimal Threshold: {opt_result['optimal_threshold']}")
print(f"Best F1: {opt_result['best_score']}")
print(f"\nMetrics at optimal threshold:")
for k, v in opt_result["metrics_at_threshold"].items():
    print(f"  {k}: {v}")


### Threshold Analysis

The optimal threshold is lower than 0.5, which makes sense for churn prediction:
- **Lower threshold** = catch more churners (higher recall)
- **Higher threshold** = fewer false alarms (higher precision)
- For retention campaigns, we generally prefer **higher recall** (catch more at-risk customers)

## 8. Save Models


In [ ]:
save_models(results, best_name, best_model)
# Also save preprocessing artifacts
joblib.dump(scaler, MODELS_DIR / "scaler.joblib")
import json
with open(MODELS_DIR / "feature_names.json", "w") as f:
    json.dump(feature_names, f)
X_test.to_csv(MODELS_DIR / "X_test.csv", index=False)
y_test.to_csv(MODELS_DIR / "y_test.csv", index=False)
print("All models and artifacts saved.")


## Summary

| Model | ROC-AUC | F1 |
|-------|---------|-----|
| Results shown above | | |

### Key Decisions
1. **Best model selected by ROC-AUC** (overall discrimination ability)
2. **Threshold optimized for F1** (balance precision and recall)
3. **All 5 models saved** for comparison and ensemble potential
4. **Class imbalance handled** via class weights and threshold optimization
